In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch_geometric

import sys
sys.path.append('./..')
    
from ld_gcn import network, loader, plotting, preprocessing, testing, error, training, utils, initialization, loss
from IPython.display import HTML

import numpy as np

# Define PDE problem

In [ ]:
pde_problem = 13
problem_name, variable, mu_space, n_param, dim_pde, n_comp, n_sim, HyperParams = utils.prepare_HyperParams(pde_problem)

# Initialize device and set reproducibility

In [ ]:
device = initialization.initialize(HyperParams)

# Load dataset

In [ ]:
dataset_dir = '../dataset/'+problem_name+'_unstructured.mat'
dataset = loader.LoadDataset(dataset_dir, variable, dim_pde, n_comp)

In [ ]:
n_snap2keep = len(mu_space[1])
delete_first_n = 1

dataset, mu_space = preprocessing.delete_initial_condition(dataset, mu_space, n_comp, n_snap2keep, n_delete=delete_first_n, shrink_param_space=True)
params = utils.create_param_list(mu_space, device)

In [ ]:
graph_loader, train_loader, test_loader, \
    val_loader, scaler_all, scaler_test, xyz, VAR_all, VAR_val, VAR_test, \
        train_trajectories, val_trajectories, test_trajectories, params_train, params_test = preprocessing.graphs_dataset(dataset, HyperParams, params)

# Define the architecture

In [ ]:
RecNet = network.RecNet(HyperParams)
RecNet = RecNet.to(device)
DynNet = network.DynNet(HyperParams)
DynNet = DynNet.to(device)

torch.set_default_dtype(torch.float32)

optimizer = 'ADAM' # 'ADAM' or 'LBFGS'

if optimizer == 'ADAM':
    optimizer = torch.optim.Adam([
        {'params': DynNet.parameters()},
        {'params': RecNet.parameters()}
      ],
      lr=HyperParams.learning_rate,
      weight_decay=HyperParams.weight_decay
    )

elif optimizer == 'LBFGS':
    optimizer = torch.optim.LBFGS(
        list(DynNet.parameters())+list(RecNet.parameters()),
        lr = 1.,
        max_iter = HyperParams.max_epochs,
        max_eval = None,
        tolerance_grad = 1e-07,
        tolerance_change = 1e-09,
        history_size = 30,
        line_search_fn = 'strong_wolfe',
    )

scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=HyperParams.miles, gamma=HyperParams.gamma)

In [ ]:
compile = False

if compile:
    try:
        import torch._dynamo
        torch._dynamo.config.suppress_errors = True
        RecNet = torch.compile(RecNet)
        DynNet = torch.compile(DynNet)
        print('The networks have been compiled successfully')
    except:
        print('Not possible to compile the networks')

In [ ]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters in dynnet:", count_trainable_params(DynNet))
print("Trainable parameters in recnet:", count_trainable_params(RecNet))

# Train or load a pre-trained network

In [ ]:
# To reduce memory consumption on GPU:
params = params.to("cpu")
VAR_all = VAR_all.to("cpu")
VAR_val = VAR_val.to("cpu")
VAR_test = VAR_test.to("cpu")

if device=='cuda':
    torch.cuda.empty_cache()

In [ ]:
load = True
train = False

if load:
    try:
        RecNet.load_state_dict(torch.load(HyperParams.net_dir+HyperParams.net_name+HyperParams.net_run+'_decoder.pt', map_location=torch.device('cpu')))
        DynNet.load_state_dict(torch.load(HyperParams.net_dir+HyperParams.net_name+HyperParams.net_run+'_dyn.pt', map_location=torch.device('cpu')))
        print('Loading saved network')
    except FileNotFoundError:
        print('Not possible to load the network')
        train = True

if train:
    training.train(RecNet, DynNet, optimizer, device, scheduler, train_loader, test_loader, HyperParams, params_train, params_test, loss.physics_loss)

# Evaluate the model

In [ ]:
RecNet = RecNet.to("cpu")
DynNet = DynNet.to("cpu")
params = params.to("cpu")

vars = "GCA-ROM"
VAR_train = VAR_all[train_trajectories,:,:]

In [ ]:
results, latents = testing.evaluate(VAR_all, RecNet, DynNet, graph_loader, params, HyperParams)
results_test, latents_test = testing.evaluate(VAR_test, RecNet, DynNet, test_loader, params_test, HyperParams)
results_train, latents_ = testing.evaluate(VAR_train, RecNet, DynNet, train_loader, params_train, HyperParams)
results_val = results[val_trajectories,:,:]

# Compute the errors

In [ ]:
vars = problem_name

error_abs, norm = error.compute_error(results, VAR_all, scaler_all, True)
error_abs_test, norm_test = error.compute_error(results_test, VAR_test, scaler_all, True)
error_abs_train, norm_train = error.compute_error(results_train, VAR_train, scaler_all, True)
error_abs_val, norm_val = error.compute_error(results_val, VAR_val, scaler_all, True)

print('\nERRORS ON THE TRAINING SET:')
print(f'NRMSE (train): {error_abs_train}')
print('\nERRORS ON THE TEST SET:')
print(f'NRMSE (test): {error_abs_test}')
print('\nERRORS ON THE VALIDATION DATASET:')
print(f'NRMSE (validation): {error_abs_val}')
print('\nERRORS ON THE WHOLE DATASET:')
print(f'NRMSE (train): {error_abs}')

In [ ]:
n_snapshots = n_snap2keep - delete_first_n
print(f'Indices of test trajectories: {(np.array(test_trajectories)/(n_snapshots))[::n_snapshots]}')

# Plot the results

In [ ]:
plotting.plot_loss(HyperParams)

SAMPLE = 3
plotting.plot_latent_time(HyperParams, SAMPLE, latents, params, n_sim)

component = 2
plotting.plot_latent_component(HyperParams, component, latents, params, n_sim)

total_times = n_snap2keep - delete_first_n
SNAP = 20

SAMPLE = 10
for SNAP in (1,6):
    plotting.plot_fields(SAMPLE, SNAP, results, scaler_all, HyperParams, dataset, params, lid_driven=True, N_DIGITS_TO_PLOT=9)